# DeepHybrid Benchmark Evaluation

Evaluate trained DeepACO JSSP models across the full JobShopLib benchmark catalog,
choose benchmark families or exact instances, load weights from `.pt` or Lightning `.ckpt`,
and plot comparison charts.


In [ ]:
# Optional Colab bootstrap
import os
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB and not Path('/content/DeepHybrid').exists():
    !git clone https://github.com/pranceraz/DeepHybrid.git

if IN_COLAB:
    os.chdir('/content/DeepHybrid')

# Core deps for this notebook
!pip -q install rl4co[graph] torch-geometric job-shop-lib pandas matplotlib

print('Working dir:', os.getcwd())


In [ ]:
import glob
import pickle
import re
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from tensordict.tensordict import TensorDict

from job_shop_lib.benchmarking import load_all_benchmark_instances, load_benchmark_instance

from generator import MyJSSPGenerator
from my_env import OperationSelectionEnv
from init_embedding import JSSPInitEmbedding, JsspEdgeEmbedding
from aco_class import MyAntSystem
from rl4co.models.zoo.nargnn.encoder import NARGNNEncoder
from rl4co.models.zoo.deepaco.policy import DeepACOPolicy
from rl4co.models.zoo.deepaco.model import DeepACO

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', DEVICE)


In [ ]:
# ===============================
# User Config
# ===============================

# Type model paths directly here. Add more than one if you want comparisons.
MODEL_PATHS = [
    '/content/DeepHybrid/epoch=9-step=1280.ckpt',
    # '/content/drive/MyDrive/DeepHybrid/training_runs/.../checkpoints/last.ckpt',
]

# Type benchmark keywords or exact benchmark names here.
# Examples: ['ft06'], ['ft'], ['ta'], ['ft06', 'ta01']
BENCHMARK_KEYWORDS = [
    'ft06',
]

# If True, fallback to full trusted checkpoint loading for your own .ckpt files.
TRUST_CHECKPOINTS = True

MODEL_GLOB_PATTERNS = [
    '*.pt',
    'lightning_logs/version_*/checkpoints/*.ckpt',
    'training_runs/*/*.pt',
    'training_runs/*/checkpoints/*.ckpt',
    '/content/drive/MyDrive/DeepHybrid/training_runs/*/*.pt',
    '/content/drive/MyDrive/DeepHybrid/training_runs/*/checkpoints/*.ckpt',
]

# Inference controls
N_ANTS_TEST = 64
N_ITER_TEST = 10

def natural_sort_key(text: str):
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r'(\d+)', text)]


def benchmark_family(name: str) -> str:
    match = re.match(r'[A-Za-z]+', name)
    return match.group(0).lower() if match else name.lower()


ALL_BENCHMARKS = load_all_benchmark_instances()
ALL_BENCHMARK_NAMES = sorted(ALL_BENCHMARKS.keys(), key=natural_sort_key)

BENCHMARKS_BY_GROUP = defaultdict(list)
for benchmark_name in ALL_BENCHMARK_NAMES:
    BENCHMARKS_BY_GROUP[benchmark_family(benchmark_name)].append(benchmark_name)

AVAILABLE_GROUPS = sorted(BENCHMARKS_BY_GROUP.keys(), key=natural_sort_key)


def discover_model_paths():
    paths = []
    for pattern in MODEL_GLOB_PATTERNS:
        paths.extend(glob.glob(pattern))
    return sorted({str(Path(p)) for p in paths}, key=natural_sort_key)


DISCOVERED_MODEL_PATHS = discover_model_paths()


def resolve_model_paths(model_paths):
    cleaned = [str(Path(p)) for p in model_paths if str(p).strip()]
    if cleaned:
        return cleaned
    if DISCOVERED_MODEL_PATHS:
        return DISCOVERED_MODEL_PATHS
    raise RuntimeError('No model artifacts found. Set MODEL_PATHS to a .pt or .ckpt path.')


def resolve_benchmark_names(keywords):
    resolved = []
    for keyword in keywords:
        kw = keyword.strip().lower()
        if not kw:
            continue
        if kw == 'all':
            resolved.extend(ALL_BENCHMARK_NAMES)
            continue
        if kw in BENCHMARKS_BY_GROUP:
            resolved.extend(BENCHMARKS_BY_GROUP[kw])
            continue
        matches = [name for name in ALL_BENCHMARK_NAMES if name.lower().startswith(kw)]
        if matches:
            resolved.extend(matches)
            continue
        raise RuntimeError(f'No benchmark matched keyword: {keyword}')
    if not resolved:
        raise RuntimeError('No benchmarks selected. Set BENCHMARK_KEYWORDS to values like ft06, ft, ta, or all.')
    return sorted(set(resolved), key=natural_sort_key)


print('Available benchmark groups:', ', '.join(['all'] + AVAILABLE_GROUPS))
print('Total benchmarks in job_shop_lib:', len(ALL_BENCHMARK_NAMES))
print('Auto-discovered model artifacts:')
for p in DISCOVERED_MODEL_PATHS:
    print(' -', p)


In [ ]:
# ===============================
# Helpers: model loading + conversion
# ===============================

class FixedBatchGenerator:
    """Compatibility stub so notebook-saved Lightning checkpoints can unpickle."""
    def __init__(self, base_generator=None, batch_size=None):
        self.base = base_generator
        self.batch_size = batch_size


SAFE_GLOBALS = [
    OperationSelectionEnv,
    MyJSSPGenerator,
    FixedBatchGenerator,
    JSSPInitEmbedding,
    JsspEdgeEmbedding,
    MyAntSystem,
]


def load_state_dict_from_artifact(path: str) -> dict:
    try:
        with torch.serialization.safe_globals(SAFE_GLOBALS):
            obj = torch.load(path, map_location='cpu', weights_only=True)
    except Exception as weights_error:
        if not TRUST_CHECKPOINTS:
            raise RuntimeError(
                f'weights_only load failed for {path}. Set TRUST_CHECKPOINTS=True if this is your own checkpoint.'
            ) from weights_error
        try:
            obj = torch.load(path, map_location='cpu', weights_only=False)
        except Exception as full_error:
            raise RuntimeError(
                f'Failed to load checkpoint {path} with both weights_only=True and weights_only=False.'
            ) from full_error
    if isinstance(obj, dict) and 'state_dict' in obj:
        return obj['state_dict']
    if isinstance(obj, dict):
        return obj
    raise ValueError(f'Unsupported artifact format: {path}')


def infer_embed_dim_from_state_dict(sd: dict) -> int:
    # scalar_embed.weight shape: [embed_dim, num_feats]
    key = 'policy.encoder.init_embedding.scalar_embed.weight'
    if key not in sd:
        raise KeyError(f'Missing key in state dict: {key}')
    return int(sd[key].shape[0])


def op_machine_from_jobshoplib_operation(op):
    # job_shop_lib Operation exposes either machine_id or machines
    if hasattr(op, 'machine_id') and op.machine_id is not None and int(op.machine_id) >= 0:
        return int(op.machine_id)

    machines = getattr(op, 'machines', None)
    if isinstance(machines, (list, tuple)):
        if len(machines) != 1:
            raise ValueError('Flexible operation encountered (multiple machines). This env expects classic JSSP.')
        return int(machines[0])

    return int(machines)


def instance_to_tensordict(instance) -> TensorDict:
    num_jobs = int(instance.num_jobs)
    num_machines = int(instance.num_machines)

    job_lengths = [len(job) for job in instance.jobs]
    total_ops = sum(job_lengths)

    start = []
    end = []
    cursor = 0
    for jl in job_lengths:
        start.append(cursor)
        end.append(cursor + jl - 1)
        cursor += jl

    proc_times = torch.zeros((1, num_machines, total_ops), dtype=torch.float32)
    machine_id = torch.empty((1, total_ops), dtype=torch.long)
    pos_in_job = torch.empty((1, total_ops), dtype=torch.float32)

    op_idx = 0
    for job in instance.jobs:
        for pos, op in enumerate(job):
            m = op_machine_from_jobshoplib_operation(op)
            d = int(op.duration)
            proc_times[0, m, op_idx] = float(d)
            machine_id[0, op_idx] = m
            pos_in_job[0, op_idx] = float(pos)
            op_idx += 1

    td = TensorDict(
        {
            'start_op_per_job': torch.tensor([start], dtype=torch.long),
            'end_op_per_job': torch.tensor([end], dtype=torch.long),
            'proc_times': proc_times,
            'pad_mask': torch.zeros((1, total_ops), dtype=torch.bool),
            'pos_in_job': pos_in_job,
            'machine_id': machine_id,
        },
        batch_size=[1],
    )

    return td


class SingleInstanceGenerator:
    def __init__(self, instance_td: TensorDict):
        self.instance_td = instance_td
        self.num_jobs = int(instance_td['start_op_per_job'].shape[-1])
        self.num_mas = int(instance_td['proc_times'].shape[1])
        self.n_ops_max = int(instance_td['proc_times'].shape[2])

    def __call__(self, batch_size=None):
        if batch_size is None:
            return self.instance_td.clone()
        if isinstance(batch_size, int):
            batch_size = [batch_size]
        b = int(batch_size[0])
        if b != 1:
            raise ValueError('SingleInstanceGenerator only supports batch_size=1 for benchmark eval.')
        return self.instance_td.clone()


def build_model_for_instance(instance, state_dict: dict):
    embed_dim = infer_embed_dim_from_state_dict(state_dict)
    num_machines = int(instance.num_machines)

    td_instance = instance_to_tensordict(instance)
    env = OperationSelectionEnv(SingleInstanceGenerator(td_instance))

    init_emb = JSSPInitEmbedding(embed_dim=embed_dim, num_machines=num_machines)
    edge_emb = JsspEdgeEmbedding(embed_dim=embed_dim)
    encoder = NARGNNEncoder(embed_dim=embed_dim, init_embedding=init_emb, edge_embedding=edge_emb)

    policy = DeepACOPolicy(
        encoder=encoder,
        env_name='tsp',
        n_ants=dict(train=10, val=20, test=N_ANTS_TEST),
        n_iterations=dict(train=1, val=5, test=N_ITER_TEST),
        aco_class=MyAntSystem,
    )

    model = DeepACO(
        env=env,
        policy=policy,
        train_with_local_search=False,
    )

    model.load_state_dict(state_dict, strict=True)
    model.to(DEVICE)
    model.eval()
    return model, env


In [ ]:
# ===============================
# Helpers: rollout + metrics
# ===============================

@torch.no_grad()
def evaluate_model_makespan(model, env) -> float:
    td = env.reset(batch_size=[1])
    td = td.to(DEVICE)
    out = model.policy(td, env, phase='test')
    reward = out['reward']
    if reward.numel() > 1:
        reward = reward.mean()
    return float((-reward).item())


def rollout_baseline_makespan(env, strategy='first_valid', seed=0) -> float:
    g = torch.Generator(device='cpu')
    g.manual_seed(seed)

    td = env.reset(batch_size=[1])
    max_steps = int(env.num_ops * 3)

    for _ in range(max_steps):
        if env._get_done(td).all():
            break

        mask = td['action_mask'][0]
        valid = torch.where(mask)[0]
        if valid.numel() == 0:
            break

        if strategy == 'first_valid':
            a = valid[0]
        elif strategy == 'random':
            idx = torch.randint(0, valid.numel(), (1,), generator=g).item()
            a = valid[idx]
        else:
            raise ValueError(f'Unknown strategy: {strategy}')

        td['action'] = a.unsqueeze(0).to(td.device)
        td = env.step(td)['next']

    makespan = td['machine_available'].max().item()
    return float(makespan)


def model_label(path: str) -> str:
    p = Path(path)
    if p.suffix == '.ckpt':
        return f'ckpt:{p.parent.parent.name}/{p.name}'
    return f'pt:{p.name}'


In [ ]:
# ===============================
# Run Evaluation
# ===============================

MODEL_PATHS_TO_EVAL = resolve_model_paths(MODEL_PATHS)
BENCHMARK_NAMES_TO_EVAL = resolve_benchmark_names(BENCHMARK_KEYWORDS)

print('Evaluating benchmarks:')
for name in BENCHMARK_NAMES_TO_EVAL:
    print(' -', name)

print('\nUsing model artifacts:')
for path in MODEL_PATHS_TO_EVAL:
    print(' -', path)

instances = []
for name in BENCHMARK_NAMES_TO_EVAL:
    inst = load_benchmark_instance(name)
    instances.append(inst)

rows = []

for model_path in MODEL_PATHS_TO_EVAL:
    print(f'\nLoading model: {model_path}')
    sd = load_state_dict_from_artifact(model_path)

    for inst in instances:
        inst_name = inst.name

        try:
            model, env = build_model_for_instance(inst, sd)
        except Exception as e:
            print(f'  - skip {inst_name}: {e}')
            continue

        model_ms = evaluate_model_makespan(model, env)
        first_ms = rollout_baseline_makespan(env, strategy='first_valid')
        rand_ms = rollout_baseline_makespan(env, strategy='random', seed=42)

        optimum = inst.metadata.get('optimum') if isinstance(inst.metadata, dict) else None

        rows += [
            {'instance': inst_name, 'solver': model_label(model_path), 'makespan': model_ms, 'optimum': optimum},
            {'instance': inst_name, 'solver': 'baseline:first_valid', 'makespan': first_ms, 'optimum': optimum},
            {'instance': inst_name, 'solver': 'baseline:random', 'makespan': rand_ms, 'optimum': optimum},
        ]

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError('No successful evaluations. Check model/instance dimension compatibility.')

df = df.sort_values(['instance', 'solver']).reset_index(drop=True)
df


In [ ]:
# ===============================
# Plot 1: Makespan Comparison
# ===============================

fig, ax = plt.subplots(figsize=(12, 5))

pivot = df.pivot_table(index='instance', columns='solver', values='makespan', aggfunc='mean')
pivot.plot(kind='bar', ax=ax)

ax.set_title('Benchmark Makespan Comparison')
ax.set_ylabel('Makespan (lower is better)')
ax.set_xlabel('Instance')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# ===============================
# Plot 2: Percent Gap to Optimum (if available)
# ===============================

df_gap = df.dropna(subset=['optimum']).copy()
if not df_gap.empty:
    df_gap['gap_pct'] = (df_gap['makespan'] - df_gap['optimum']) / df_gap['optimum'] * 100.0

    fig, ax = plt.subplots(figsize=(12, 5))
    gap_pivot = df_gap.pivot_table(index='instance', columns='solver', values='gap_pct', aggfunc='mean')
    gap_pivot.plot(kind='bar', ax=ax)

    ax.axhline(0.0, color='black', linewidth=1)
    ax.set_title('Percent Gap to Benchmark Optimum')
    ax.set_ylabel('Gap (%) lower is better')
    ax.set_xlabel('Instance')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print('No optimum metadata available for selected instances.')


## Notes

- Set `MODEL_PATHS` directly to the `.pt` or `.ckpt` files you want to compare.
- Set `BENCHMARK_KEYWORDS` to exact names like `ft06` or prefixes/groups like `ft`, `ta`, `la`, `abz`, or `all`.
- If `MODEL_PATHS` is empty, the notebook falls back to auto-discovered local and Drive artifacts.
- If `TRUST_CHECKPOINTS=True`, the notebook will use a more permissive fallback loader for your own Lightning checkpoints.
- This notebook currently evaluates one instance at a time (`batch_size=1`).
- If a benchmark is skipped, it usually means model architecture dimensions do not match that instance.
  Example: a model trained for `6x6` will not load for `10x10`/`15x15` without retraining.
- `.ckpt` and `.pt` are both supported. `.ckpt` loads `state_dict` automatically.
